<a href="https://colab.research.google.com/github/komorizmh07/1120/blob/main/%E6%A9%9F%E6%A2%B0%E5%AD%A6%E7%BF%92_1120.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [3]:
train=pd.read_csv("train.csv") #これはcolabのファイル内に事前に保存しておいたやつです
test=pd.read_csv("test.csv")
sub=pd.read_csv("sample_submission.csv")

In [4]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Load data
# train = pd.read_csv("train.csv")
# test = pd.read_csv("test.csv")

# Drop unneeded columns and handle missing values
train = train.drop(columns=["Unnamed: 12", "id"])  # Dropping unnecessary columns

# Find common columns between train and test (excluding "DIC" from train)
common_columns = train.drop(columns=["DIC"]).columns.intersection(test.columns)

# Select the common columns for both train and test
X = train[common_columns]
y = train["DIC"]
test = test[common_columns]

# Replace NaNs with the column mean
# X.fillna(X.mean(), inplace=True)
# test.fillna(test.mean(), inplace=True)
X.fillna(X.mean())
test.fillna(test.mean())

# Normalize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
test_scaled = scaler.transform(test)

# Split the training data as before
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [5]:
from torch.utils.data import Dataset, DataLoader

class OceanChemistryDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = OceanChemistryDataset(X_train, y_train)
val_dataset = OceanChemistryDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


In [10]:
class MLPModel(nn.Module):
    def __init__(self, input_size):
        super(MLPModel, self).__init__()

        #drop層は今回のデータ量が少ないので加えると学習が進みづらかった。->加えていません。

        self.fc1 = nn.Linear(input_size, 64)
        self.relu1 = nn.ReLU()

        self.fc2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()

        self.fc3 = nn.Linear(32, 16)
        self.relu3 = nn.ReLU()

        self.fc4 = nn.Linear(16, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)

        x = self.fc2(x)
        x = self.relu2(x)

        x = self.fc3(x)
        x = self.relu3(x)

        x = self.fc4(x)
        return x

# Initialize the model
model = MLPModel(input_size=X_train.shape[1])
print(X_train.shape)

(1163, 15)


In [11]:
import torch.optim as optim

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001, weight_decay = 1e-5)

# Training function
def train_model(model, train_loader, val_loader, epochs = 7000):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs.squeeze(), y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        val_loss = 0.0
        model.eval()
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                outputs = model(X_batch)
                loss = criterion(outputs.squeeze(), y_batch)
                val_loss += loss.item()

        if epoch % 100 == 0:
            print(f"Epoch {epoch + 1} / {epochs}, Train Loss: {running_loss / len(train_loader)}, Validation Loss: {val_loss / len(val_loader)}")

# Train the model
train_model(model, train_loader, val_loader, epochs = 7000)


Epoch 1 / 7000, Train Loss: 4645877.105263158, Validation Loss: 4618333.3
Epoch 101 / 7000, Train Loss: 8790.208444695723, Validation Loss: 9716.7158203125
Epoch 201 / 7000, Train Loss: 1677.0951120476973, Validation Loss: 1676.0630615234375
Epoch 301 / 7000, Train Loss: 902.6344475997122, Validation Loss: 843.5784545898438
Epoch 401 / 7000, Train Loss: 500.9612458881579, Validation Loss: 554.528466796875
Epoch 501 / 7000, Train Loss: 282.725117733604, Validation Loss: 366.9613342285156
Epoch 601 / 7000, Train Loss: 182.73524033395867, Validation Loss: 226.68655090332032
Epoch 701 / 7000, Train Loss: 111.80375350149055, Validation Loss: 150.4619369506836
Epoch 801 / 7000, Train Loss: 81.19970140959087, Validation Loss: 97.67065048217773
Epoch 901 / 7000, Train Loss: 60.35802821109169, Validation Loss: 74.62448501586914
Epoch 1001 / 7000, Train Loss: 47.318547800967565, Validation Loss: 59.29675979614258
Epoch 1101 / 7000, Train Loss: 47.25968370939556, Validation Loss: 64.4278495788574

In [14]:
# Convert the test set into a torch tensor
test_tensor = torch.tensor(test_scaled, dtype=torch.float32)

# Set the model to evaluation mode
model.eval()

# Make predictions
with torch.no_grad():
    predictions = model(test_tensor).squeeze().numpy()

# Prepare submission
submission = pd.DataFrame({"id": range(1455, 1455 + len(predictions)), "DIC": predictions})
submission.to_csv("submission_3.csv", index=False)


In [15]:
# score : 5.30029
# name: 小森陽向
# id: 62507575